# Week 09 — Budget pacing and CPA control

**Goal.** Build a controller that spends a budget smoothly and hits a target CPA, then break it with delayed conversion data and fix it.

**Deliverable.** A pacing + target-CPA controller and the overspend-under-delayed-data analysis. This is the startup prototype.

**Rough shape of the week.** 2h reading (Smart Pacing) · 6h building · 1h write-up.

---
### Ground rules (they apply every week)

1. **Beat a dumb baseline or it didn't happen.** Logistic regression or the global mean.
   Log the baseline in the same table as the fancy model.
2. **Split by time, never at random.** `split.time_split` — and call
   `split.check_no_leakage` so the assertion, not your memory, enforces it.
3. **Log every run** with `registry.log_result(...)`, including the ones that lost.
   The losing runs are what make the write-up honest.
4. **Write the finding down** in this week's `README.md` while it is fresh.

### Reading

PDFs are in `papers/` next to this notebook — see `papers/README.md`.

In [ ]:
import sys, warnings
sys.path.insert(0, "..")
warnings.filterwarnings("ignore", category=FutureWarning)

%load_ext autoreload
%autoreload 2

import numpy as np, pandas as pd, matplotlib.pyplot as plt
from adslab import data, metrics, plots, split, registry, encoders, calibration

plots.use_style()
pd.set_option("display.width", 140, "display.max_columns", 60)
print("harness ready")

## The week that matters most to you

Everything here feeds the budget-control idea. The experiment in section 5 is the
technical heart of it: **a CPA controller fed delayed conversion data overspends, and it
overspends in a specific, predictable, correctable way.** You saw this from the inside at
Google. This notebook reproduces it from the outside on open data, which is what makes it
something you can show anyone.

Build on Week 8's auction simulator and Week 4's delay model.

In [ ]:
df = data.add_attribution_derived(data.load_attribution())

hourly = df.groupby(df.hour_of_day).size()
fig, ax = plt.subplots()
ax.bar(hourly.index, hourly.values)
ax.set_xlabel("hour of day"); ax.set_ylabel("impressions")
ax.set_title("Traffic shape — what the pacer has to spend against")
print(plots.save(fig, 9, "traffic_by_hour"))

## 1. Budget pacing

Spend a daily budget smoothly against non-uniform traffic. Two families:

- **Probabilistic throttling**: participate in a fraction $\theta$ of auctions.
- **Bid modulation**: participate always, scale the bid by $\mu$.

Smart Pacing argues for the second on quality grounds — throttling drops good and bad
impressions indiscriminately, while modulating keeps you in the auctions you value most.
Implement both and show the difference in *what you bought*, not just in spend curve
smoothness.

The controller itself: PID on the error between actual and target cumulative spend.
Start with P only; add I when you see steady-state offset; add D last, if ever.

In [ ]:
class PacingController:
    def __init__(self, daily_budget, kp=0.5, ki=0.05, kd=0.0):
        # TODO
        raise NotImplementedError

    def step(self, spent_so_far, target_so_far, dt):
        """Return the bid multiplier for the next interval."""
        raise NotImplementedError

## 2. Target CPA

Now the outer loop: adjust the bid multiplier to hit a target cost per acquisition.

$$\text{CPA} = \frac{\text{spend}}{\text{conversions}}$$

Bid up and you win more and pay more per win; the relationship is nonlinear and noisy at
low conversion volume. Tune on a simulated week and record the settling time.

Note the sample-size problem that makes this hard in reality: at a 0.2% CVR and a few
thousand impressions an hour, an hourly CPA estimate is built on single-digit
conversions. The controller is mostly reacting to noise. Show that — plot the hourly CPA
estimate with its confidence interval next to the controller's response.

In [ ]:
class TargetCPAController:
    def __init__(self, target_cpa, kp=0.3, ki=0.02):
        # TODO
        raise NotImplementedError

## 3. The good case

Instant conversion reporting. Run a simulated week and confirm the controller converges:
spend tracks target, CPA settles near the goal. Establish that it works before you break
it, or you will not know which failure is which.

In [ ]:
# TODO

## 4. Break it — delayed reporting

**The experiment.** Feed the controller conversions with the Week 4 lag distribution
attached instead of instantly.

What should happen: early in the day, observed conversions are far below eventual
conversions, so the measured CPA looks terrible, so the controller bids *down* — or, if
it is a spend-pacing controller with budget left over, it bids *up* to spend the budget
and buys traffic it should not. Either way it is steering on a signal that is
systematically wrong in a known direction.

Quantify it: overspend as a percentage of budget, and realised CPA vs target, as a
function of the delay distribution's median. Sweep the median from 0 to 48 hours and
plot. **That curve is the pitch.**

In [ ]:
# TODO: inject delays, sweep median delay, plot overspend %

## 5. Fix it

Use the Week 4 model to estimate *eventual* conversions from observed ones, and feed the
lag-corrected CPA to the controller. Re-run the sweep.

Then be honest about the residual: the correction has its own error, and a controller
that trusts a corrected estimate too much has a new failure mode. Show the corrected
curve next to the naive one and mark where correction stops helping.

In [ ]:
# fig, ax = plt.subplots()
# ... overspend vs median delay: naive vs lag-corrected
# print(plots.save(fig, 9, "overspend_vs_delay"))

---
## Log the results

Every model you tried, including the baseline and including the failures. `notes` is the
one sentence you would say out loud about the run — future-you assembles the write-up
from these, so write it now while you still remember why the run mattered.

In [ ]:
# registry.log_result(
#     week=9,
#     model="lightgbm_hashed_2^18",
#     metrics=metrics.evaluate(y_test, p_test),
#     dataset="attribution",
#     params=dict(n_bits=18, num_leaves=63, lr=0.05),
#     notes="beats LR by 0.011 AUC; most of the gain is from cat3 x cat7 interactions",
# )

print(registry.to_markdown(week=9))

---
## Write it up

Open `README.md` in this folder and fill in the three sections. Keep it to a page.

- **What I built** — one paragraph, no code.
- **What the numbers say** — paste the table above; say which comparison is the honest one.
- **What surprised me** — the part worth reading. If nothing surprised you, you probably
  did not stress the model hard enough.

Then commit:

```bash
git add week09_* results/
git commit -m "week 09: <the finding, not the task>"
```